# Cisco CX CDC: Ibis / DuckDB (Portable)

**Source**: S3 Parquet (10M rows)  
**CDC**: Hash-based diff with Ibis (DuckDB backend)  
**Target**: Snowflake table  
**Runs on**: Snowflake SPCS and EC2/Docker identically  

Ibis compiles the same DataFrame API to DuckDB (local) or Snowflake (cloud).

In [ ]:
import importlib
required = [('ibis-framework', 'ibis'), ('duckdb', 'duckdb'), ('pyarrow', 'pyarrow'), ('s3fs', 's3fs')]
missing = []
for pkg, mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing + ['-q'])
    print(f"Installed: {missing}")
else:
    print("All packages available.")

In [ ]:
import os

def get_snowflake_connection():
    """Connect to Snowflake - works in SPCS or on-prem."""
    try:
        from snowflake.snowpark.context import get_active_session
        session = get_active_session()
        print("Connected via Snowflake SPCS")
        return session, "SPCS"
    except:
        pass
    import snowflake.connector
    conn = snowflake.connector.connect(connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME", "default"))
    print(f"Connected locally to {conn.account}")
    return conn, "LOCAL"

In [ ]:
import os, time
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

S3_BUCKET = "cisco-cx-cdc-pilot"
S3_PREV = "s3://cisco-cx-cdc-pilot/cdc_data/prev_snapshot/"
S3_CURR = "s3://cisco-cx-cdc-pilot/cdc_data/curr_snapshot/"

COMPARE_COLS = ['software_version', 'cpu_utilization', 'memory_utilization',
                'critical_bugs_count', 'contract_status', 'ip_address']

def detect_runtime():
    try:
        from snowflake.snowpark.context import get_active_session
        get_active_session()
        return "SPCS"
    except:
        return "LOCAL"

RUNTIME = detect_runtime()
print(f"Runtime: {RUNTIME}")

if RUNTIME == "SPCS":
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Reading from Snowflake External Stage (@CDC_S3_STAGE)...")
    t0 = time.time()
    prev_pdf = session.sql("""
        SELECT $1:device_id::VARCHAR AS device_id,
               $1:customer_id::VARCHAR AS customer_id,
               $1:software_version::VARCHAR AS software_version,
               $1:cpu_utilization::FLOAT AS cpu_utilization,
               $1:memory_utilization::FLOAT AS memory_utilization,
               $1:critical_bugs_count::INTEGER AS critical_bugs_count,
               $1:contract_status::VARCHAR AS contract_status,
               $1:ip_address::VARCHAR AS ip_address,
               $1:product_family::VARCHAR AS product_family
        FROM @CISCO_CX_PILOT.LANDING_ZONE.CDC_S3_STAGE/prev_snapshot/
    """).to_pandas()
    prev_pdf.columns = [c.lower() for c in prev_pdf.columns]
    print(f"  Read {len(prev_pdf):,} rows (prev) in {time.time()-t0:.1f}s")
    t0 = time.time()
    curr_pdf = session.sql("""
        SELECT $1:device_id::VARCHAR AS device_id,
               $1:customer_id::VARCHAR AS customer_id,
               $1:software_version::VARCHAR AS software_version,
               $1:cpu_utilization::FLOAT AS cpu_utilization,
               $1:memory_utilization::FLOAT AS memory_utilization,
               $1:critical_bugs_count::INTEGER AS critical_bugs_count,
               $1:contract_status::VARCHAR AS contract_status,
               $1:ip_address::VARCHAR AS ip_address,
               $1:product_family::VARCHAR AS product_family
        FROM @CISCO_CX_PILOT.LANDING_ZONE.CDC_S3_STAGE/curr_snapshot/
    """).to_pandas()
    curr_pdf.columns = [c.lower() for c in curr_pdf.columns]
    print(f"  Read {len(curr_pdf):,} rows (curr) in {time.time()-t0:.1f}s")
else:
    import pyarrow.fs as pafs
    print("Reading directly from S3...")
    fs = pafs.S3FileSystem(region='us-east-1')
    t0 = time.time()
    prev_pdf = pq.read_table(f"{S3_BUCKET}/cdc_data/prev_snapshot", filesystem=fs).to_pandas()
    print(f"  Read {len(prev_pdf):,} rows (prev) in {time.time()-t0:.1f}s")
    t1 = time.time()
    curr_pdf = pq.read_table(f"{S3_BUCKET}/cdc_data/curr_snapshot", filesystem=fs).to_pandas()
    print(f"  Read {len(curr_pdf):,} rows (curr) in {time.time()-t1:.1f}s")

print(f"Previous: {len(prev_pdf):,} | Current: {len(curr_pdf):,}")

In [ ]:
import ibis
ibis.options.interactive = True
print(f'Ibis {ibis.__version__}')

t_start = time.time()

t0 = time.time()
con = ibis.duckdb.connect()
prev_tbl = con.create_table('prev_snapshot', prev_pdf)
curr_tbl = con.create_table('curr_snapshot', curr_pdf)
t_create = time.time() - t0
print(f'Tables loaded into DuckDB: {t_create:.1f}s')

t0 = time.time()
compare_expr_p = ibis.literal('|').join([prev_tbl[c].cast('string') for c in COMPARE_COLS])
prev_hashed = prev_tbl.mutate(_hash=compare_expr_p.hash())

compare_expr_c = ibis.literal('|').join([curr_tbl[c].cast('string') for c in COMPARE_COLS])
curr_hashed = curr_tbl.mutate(_hash=compare_expr_c.hash())
t_hash = time.time() - t0
print(f'Hash expressions defined: {t_hash:.1f}s')

t0 = time.time()
prev_keys = prev_hashed.select('device_id', _hash_prev=prev_hashed._hash)
curr_keys = curr_hashed.select('device_id', _hash_curr=curr_hashed._hash)

merged = curr_keys.outer_join(prev_keys, 'device_id')

n_inserts = merged.filter(merged._hash_prev.isnull()).count().execute()
n_deletes = merged.filter(merged._hash_curr.isnull()).count().execute()
both = merged.filter(merged._hash_curr.notnull() & merged._hash_prev.notnull())
n_updates = both.filter(both._hash_curr != both._hash_prev).count().execute()
t_cdc = time.time() - t0

total_cdc = time.time() - t_start
print(f'\n=== Ibis (DuckDB backend) CDC Results ===')
print(f'  Table load:             {t_create:.1f}s')
print(f'  Hash + Join + Classify: {t_cdc:.1f}s')
print(f'  TOTAL CDC:              {total_cdc:.1f}s')
print(f'  Inserts: {n_inserts:,} | Updates: {n_updates:,} | Deletes: {n_deletes:,}')

In [ ]:
t0 = time.time()
sf_conn, runtime = get_snowflake_connection()

insert_ids = merged.filter(merged._hash_prev.isnull()).select('device_id')
update_ids = both.filter(both._hash_curr != both._hash_prev).select('device_id')
all_change_ids = ibis.union(insert_ids, update_ids)

upsert_data = curr_tbl.filter(curr_tbl.device_id.isin(all_change_ids.device_id)).execute()
upsert_data.columns = [c.upper() for c in upsert_data.columns]

if runtime == "LOCAL":
    from snowflake.connector.pandas_tools import write_pandas
    sf_conn.cursor().execute("USE DATABASE CISCO_CX_PILOT")
    sf_conn.cursor().execute("USE SCHEMA PROCESSED")
    sf_conn.cursor().execute("CREATE TABLE IF NOT EXISTS IBIS_CDC_RESULT (DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR, CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER, CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)")
    sf_conn.cursor().execute("TRUNCATE TABLE IBIS_CDC_RESULT")
    success, nchunks, nrows, _ = write_pandas(sf_conn, upsert_data, 'IBIS_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED')
    print(f'Wrote {nrows:,} rows to Snowflake in {time.time()-t0:.1f}s')
else:
    session = sf_conn
    session.sql("USE DATABASE CISCO_CX_PILOT").collect()
    session.sql("USE SCHEMA PROCESSED").collect()
    session.write_pandas(upsert_data, 'IBIS_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED', overwrite=True)
    print(f'Wrote {len(upsert_data):,} rows to Snowflake in {time.time()-t0:.1f}s')

con.disconnect()
print(f'\n=== TOTAL PIPELINE (Read + CDC + Write): {(time.time()-t0) + total_cdc:.1f}s ===')